# Lab 16 — Transformer Architecture Deep Dive

**June 12, 2017.** Eight researchers at Google Brain and Google Research upload a 15-page paper titled [*Attention Is All You Need*](https://arxiv.org/abs/1706.03762). They discard recurrence, they discard convolutions, and they build an entire sequence model out of one operation: **attention**. Every major language model of the 2020s — GPT-4, Claude, Gemini, Llama, Mistral — is a direct descendant of that one paper. The core 70 lines of Python haven't meaningfully changed in 8 years.

By the end of this lab you will have built a real transformer from four basic operations (matrix multiply, softmax, layernorm, addition), trained it on GPU, and used it to generate text. No `nn.Transformer`, no `F.scaled_dot_product_attention` — you'll write every piece.

### Canonical references (read alongside)

- **[Attention Is All You Need (Vaswani et al., 2017)](https://arxiv.org/abs/1706.03762)** — the paper. Figure 1 is the whole architecture. Section 3.2.1 is scaled dot-product attention. Read those two first.
- **[Andrej Karpathy — Let's Build GPT from Scratch](https://www.youtube.com/watch?v=kCc8FmEb1nY)** — 2-hour video that builds up to what this lab does, with live coding. The best single educational resource on transformers ever produced.
- **[Karpathy's nanoGPT](https://github.com/karpathy/nanoGPT)** — a 300-line PyTorch GPT-2 reimplementation. Production-ready but readable. Our lab tracks its structure closely.
- **[The Illustrated Transformer (Jay Alammar)](https://jalammar.github.io/illustrated-transformer/)** — the architectural intuitions in pictures.
- **[The Annotated Transformer (Harvard NLP)](http://nlp.seas.harvard.edu/annotated-transformer/)** — the original paper with runnable PyTorch inline.

### What you'll build

A decoder-only (GPT-style) transformer, piece by piece:

1. **Scaled dot-product attention** with a causal mask — the one operation that started it all.
2. **Multi-head attention** — attention in parallel, concatenated.
3. **The Transformer Block** — attention + feed-forward + LayerNorm + residual connections.
4. **Stack blocks into a tiny GPT, train it, generate from it.**

### Scale context

| Model | Layers | d_model | Heads | Total params |
|-------|--------|---------|-------|--------------|
| *(ours, this lab)* | 2 | 64 | 4 | ~60K |
| Original Transformer Base (2017) | 6 | 512 | 8 | ~65M |
| GPT-2 (small) | 12 | 768 | 12 | 124M |
| GPT-3 (175B) | 96 | 12,288 | 96 | 175B |
| Llama 3 8B | 32 | 4,096 | 32 | 8B |

The architecture is the same at every scale — you multiply d_model, layers, and heads by constants. Understanding 60K params means understanding 175B.

---

## Step 1 — Scaled dot-product attention

Attention answers: *"For each position in my sequence, how much should I pay attention to every other position?"* It outputs, per position, a weighted mix of the values at all other positions.

### The formula (it fits on one line)

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

Three matrices per input:
- **Q (queries)** — "what am I looking for?" — one vector per position
- **K (keys)** — "what do I contain?" — one vector per position
- **V (values)** — "what do I contribute?" — one vector per position

`QK^T` gives a `T × T` score matrix: score[i, j] = how much position i cares about position j. Softmax turns the scores into probabilities. Multiply by V to get the weighted mix.

### Why the √d_k scaling?

Without it, dot products grow with the embedding dim and softmax gets too peaky (one position gets ~100% of the weight, everything else ~0%). Gradients die. The `/√d_k` keeps the variance of QK^T ~ 1 regardless of embedding size. Skipping this is one of the top-5 bugs people hit when implementing attention from scratch.

### The causal mask

For a *decoder* (generative) transformer, position i must only see positions ≤ i. Seeing the future is cheating — the model would learn to copy the next token instead of predicting it. We enforce this by **setting future-position scores to −∞ before softmax**, so they become 0 after softmax.

### Scaled dot-product attention — one function, five steps

The core operation of every transformer. Five steps, each numbered in the comments:
1. **Score** each query against each key (dot product).
2. **Scale** by √d to keep softmax sharpness bounded as d grows.
3. **Mask** future positions to -∞ (causal attention — token t can only see tokens ≤ t).
4. **Softmax** over keys turns scores into a probability distribution per query.
5. **Weighted sum** of value vectors using those probabilities.

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device('cuda')
torch.manual_seed(42)

def scaled_dot_product_attention(Q, K, V, causal=True):
    """
    Q, K, V: (B, T, D) tensors.
    Returns: output (B, T, D) and attention weights (B, T, T).
    """
    B, T, D = Q.shape
    # 1. Score each query against each key
    scores = Q @ K.transpose(-2, -1)              # (B, T, T)
    # 2. Scale by √d_k to keep softmax sharpness bounded
    scores = scores / (D ** 0.5)
    # 3. Apply causal mask — set future positions to -inf so softmax zeroes them
    if causal:
        mask = torch.triu(torch.ones(T, T, device=Q.device), diagonal=1).bool()
        scores = scores.masked_fill(mask, float('-inf'))
    # 4. Softmax over the key dimension → probability distribution per row
    weights = F.softmax(scores, dim=-1)          # (B, T, T)
    # 5. Weighted sum of values
    output = weights @ V                          # (B, T, D)
    return output, weights

### Sanity check — shapes, row sums, causality

Three invariants every attention implementation should pass:
- Output shape matches input shape (B, T, D in → B, T, D out).
- Each row of the attention weights sums to 1.0 (softmax normalization).
- With causal masking, the top-right triangle of the weights should be zero.

In [4]:
# Verify it runs and the shapes match expectation
B, T, D = 2, 8, 16
Q = torch.randn(B, T, D, device=device)
K = torch.randn(B, T, D, device=device)
V = torch.randn(B, T, D, device=device)
attention_output, attention_weights = scaled_dot_product_attention(Q, K, V, causal=True)
print(f'output shape:     {tuple(attention_output.shape)}')
print(f'weights shape:    {tuple(attention_weights.shape)}')
print(f'row sum (should be 1.0 everywhere): {attention_weights.sum(dim=-1).mean().item():.4f}')

output shape:     (2, 8, 16)
weights shape:    (2, 8, 8)
row sum (should be 1.0 everywhere): 1.0000


In [5]:
# Visual sanity: show the causal mask in action — upper triangle should be exactly 0.
print('Attention weights for the first example (rows=query, cols=key):')
print(attention_weights[0].cpu().numpy().round(2))
print()
print('Notice the upper triangle is all zeros — that is the causal mask at work.')

Attention weights for the first example (rows=query, cols=key):
[[1.   0.   0.   0.   0.   0.   0.   0.  ]
 [0.76 0.24 0.   0.   0.   0.   0.   0.  ]
 [0.23 0.39 0.38 0.   0.   0.   0.   0.  ]
 [0.27 0.13 0.19 0.41 0.   0.   0.   0.  ]
 [0.06 0.17 0.16 0.39 0.22 0.   0.   0.  ]
 [0.27 0.12 0.22 0.09 0.17 0.12 0.   0.  ]
 [0.1  0.31 0.08 0.03 0.24 0.05 0.18 0.  ]
 [0.24 0.1  0.18 0.06 0.14 0.14 0.08 0.05]]

Notice the upper triangle is all zeros — that is the causal mask at work.


In [6]:
from preporato_labs import Lab
lab = Lab('transformer-from-scratch')
lab.check(1)

OK — scaled dot-product attention shape (2, 8, 16), causal mask intact, rows sum to 1.0
STEP_PASSED


Step 1 Complete! Scroll down to continue...

True

## Step 2 — Multi-Head Attention

One attention head learns one *kind* of relationship (e.g., subject↔verb agreement). To learn many kinds of relationships in parallel, we stack **heads** — each head runs its own scaled-dot-product attention on its own slice of the embedding.

The trick: we split `d_model` into `n_heads` chunks of `d_head = d_model / n_heads`. Each head gets its own Q, K, V linear projection into its chunk, does attention there, and the outputs concatenate back to `d_model` and go through one more linear projection.

Per the original paper, this lets the model *"attend to information from different representation subspaces at different positions"*. GPT-2 has 12 heads, Llama 3 8B has 32, GPT-3 uses 96. More heads = more parallel relationships, at proportional compute cost.

### Multi-head attention — the same formula, reshaped

Split `d_model` into `n_heads` independent attention channels. Each head learns its own projection of the input into a smaller `d_head = d_model / n_heads` space, computes its own attention, and the outputs are concatenated.

Two optimizations in the implementation:
- **Fused QKV projection** — one `nn.Linear(d, 3d)` instead of three `nn.Linear(d, d)`, better memory access pattern and fewer kernel launches.
- **Transpose to `(B, n_heads, T, d_head)`** — puts the head dim next to batch so attention is batched over heads in parallel.

In [7]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        assert d_model % n_heads == 0, 'd_model must be divisible by n_heads'
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.d_model = d_model
        # Fused QKV projection for efficiency — splits into Q, K, V later
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, T, D = x.shape
        # (B, T, 3*D) -> split into Q, K, V each of shape (B, T, D)
        qkv = self.qkv(x)
        q, k, v = qkv.split(self.d_model, dim=-1)

        # Reshape to (B, n_heads, T, d_head) — put the head dim next to batch for parallelism
        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        # Scaled dot-product attention per head (same formula as Step 1, in batched form)
        scores = (q @ k.transpose(-2, -1)) / (self.d_head ** 0.5)
        mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
        scores = scores.masked_fill(mask, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        out = attn @ v                              # (B, n_heads, T, d_head)

        # Concatenate heads back to d_model
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        return self.proj(out)

### Smoke test — multi-head MHA on a tiny batch

Same API as the single-head version from step 1. Output shape should match input shape, and the param count scales with `d_model²` (the two large projections plus the bias-free Linears).

In [8]:
mha = MultiHeadAttention(d_model=32, n_heads=4).to(device)
x = torch.randn(2, 6, 32, device=device)
y = mha(x)
print(f'MHA({mha.n_heads} heads) output shape: {tuple(y.shape)}')
print(f'Trainable params: {sum(p.numel() for p in mha.parameters()):,}')

MHA(4 heads) output shape: (2, 6, 32)
Trainable params: 4,096


In [9]:
lab.check(2)

OK — MultiHeadAttention with 4 heads produces output of shape (2, 6, 32)
STEP_PASSED


Step 2 Complete! Scroll down to continue...

True

## Step 3 — The Transformer Block

A *block* is:

```
x = x + MHA( LayerNorm(x) )
x = x + FFN( LayerNorm(x) )
```

Four ingredients, each essential:

- **Multi-head attention** (just built)
- **Feed-forward network (FFN)** — two linear layers with a GELU or ReLU in between, `d_model → 4·d_model → d_model`. This is where the bulk of the parameters actually live. **Per-position** — no information flow across positions here (that's attention's job).
- **LayerNorm** — normalizes each position's vector to zero mean, unit variance. Stabilizes training. Modern transformers use **pre-LN** (before attention/FFN, shown above) rather than the original paper's post-LN; pre-LN is empirically much more stable at scale.
- **Residual connections** (the `x + ...`) — every transformer paper since 2017 has them. Without residuals, gradients die in deep networks. Introduced for ResNet (He et al., 2015); absolutely central to training networks with >10 layers.

Stacking N blocks gives you the full transformer depth. Each block lets every position look at every other position once more.

In [10]:
class FeedForward(nn.Module):
    def __init__(self, d_model: int, mult: int = 4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, mult * d_model),
            nn.GELU(),
            nn.Linear(mult * d_model, d_model),
        )

    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model)

    def forward(self, x):
        # Pre-LN variant (stable at depth)
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x

block = TransformerBlock(d_model=32, n_heads=4).to(device)
y = block(torch.randn(2, 8, 32, device=device))
print(f'Block output shape: {tuple(y.shape)}')
print(f'Block params: {sum(p.numel() for p in block.parameters()):,}')

Block output shape: (2, 8, 32)
Block params: 12,576


In [11]:
lab.check(3)

OK — TransformerBlock: shape preserved (2, 8, 32), gradients flow end-to-end
STEP_PASSED


Step 3 Complete! Scroll down to continue...

True

## Step 4 — Build a tiny GPT, train it, generate

Now we assemble a full GPT:

```
input tokens → token embedding + positional embedding
           → Transformer Block × N  
           → final LayerNorm  
           → Linear projection to vocab_size  
           → logits over next token
```

We train it character-level on a small synthetic corpus with a very clear learnable pattern. On a 3060 Ti with a 60K-param model, we can do hundreds of steps per second — loss should drop visibly in seconds.

### TinyGPT — assembly

Token + position embeddings, a stack of `TransformerBlock` layers, a final LayerNorm, and a Linear head projecting back to vocabulary logits. `generate()` is autoregressive: feed the whole context, grab the last position's logits, sample a next token, append, repeat.

In [12]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size: int, d_model: int = 64, n_heads: int = 4,
                 n_layers: int = 2, max_seq_len: int = 64):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.blocks = nn.Sequential(*[TransformerBlock(d_model, n_heads) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)    # (B, T, D)
        x = self.blocks(x)
        x = self.ln_f(x)
        return self.head(x)                          # (B, T, vocab)

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=50):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -64:]                  # crop to context
            logits = self(idx_cond)[:, -1, :]        # last position's logits
            next_id = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx

### Build a tiny corpus + char-level tokenizer

A few sentences repeated 30× — enough for the model to pick up structure in a 1-minute training run without needing a real dataset. Char-level tokenization keeps the vocab small (~40) so we can see the model memorize-then-generalize quickly.

In [13]:
# Tiny corpus — a few sentences repeated. The transformer will learn their structure quickly.
corpus = (
    'The cat sat on the mat. '
    'A quick brown fox jumps over the lazy dog. '
    'Transformers are the backbone of modern AI. '
    'Attention is all you need. '
    'Machine learning transforms software engineering.'
) * 30

# Char-level tokenization
vocab = sorted(set(corpus))
stoi = {c: i for i, c in enumerate(vocab)}
itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in corpus], dtype=torch.long, device=device)
print(f'Vocab size: {len(vocab)}  |  Corpus chars: {len(data)}')

Vocab size: 32  |  Corpus chars: 5610


### Instantiate the model

64-dim, 4 heads, 2 layers — total ~40K params. That's ~0.00003% the size of GPT-2, but it's the same architecture in miniature and enough to demonstrate every concept in this lab.

In [14]:
model = TinyGPT(vocab_size=len(vocab), d_model=64, n_heads=4, n_layers=2).to(device)
print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')

Model params: 107,776


In [15]:
# Training loop — classic LM style: predict next char at every position.
block_size = 32
batch_size = 32

def get_batch():
    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,), device=device)
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+1+block_size] for i in ix])
    return x, y

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
loss_history = []

model.train()
N_STEPS = 400
for step in range(N_STEPS):
    x, y = get_batch()
    logits = model(x)                              # (B, T, V)
    loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())
    if step % 50 == 0 or step == N_STEPS - 1:
        print(f'step {step:4d}  loss {loss.item():.3f}')

initial = sum(loss_history[:5]) / 5
final = sum(loss_history[-5:]) / 5
print(f'\nLoss: initial~{initial:.3f} -> final~{final:.3f}  ({100*(initial-final)/initial:.0f}% reduction)')

step    0  loss 3.714
step   50  loss 0.980
step  100  loss 0.178
step  150  loss 0.125
step  200  loss 0.086
step  250  loss 0.098
step  300  loss 0.087
step  350  loss 0.079
step  399  loss 0.074

Loss: initial~3.262 -> final~0.081  (98% reduction)


In [16]:
# Generate from a seed prompt — the model should produce corpus-like text
model.eval()
seed = 'The cat'
ids = torch.tensor([[stoi[c] for c in seed]], dtype=torch.long, device=device)
out_ids = model.generate(ids, max_new_tokens=120)
generated = ''.join(itos[i.item()] for i in out_ids[0])
print(f'SEED: {seed!r}')
print(f'GENERATED: {generated!r}')
print()
print('The generated text should be recognisable as corpus-shaped — words, spaces,')
print('likely actual phrases from the training set with some remixing. With only 60K')
print('params and 150 chars of unique training data, memorisation dominates creativity.')

SEED: 'The cat'
GENERATED: 'The cat sat on the mat. A quick b wn be brormps atwale quickbrat b own mown ovb own  at b ob b own own  of mat       mat. of ma'

The generated text should be recognisable as corpus-shaped — words, spaces,
likely actual phrases from the training set with some remixing. With only 60K
params and 150 chars of unique training data, memorisation dominates creativity.


In [17]:
lab.check(4)

OK — trained transformer: loss 3.262 -> 0.081 (98% reduction); 127 chars generated
STEP_PASSED


Step 4 Complete! Lab complete!

True

---

## What you just built

A real transformer, from the softmax up. Every line of code you wrote maps 1:1 to a component in GPT-4 — they differ in scale (billions vs thousands of params, thousands vs tens of heads) but not in *kind*. Understanding 60K params is understanding 175B.

## What to read next

- **[Attention Is All You Need](https://arxiv.org/abs/1706.03762)** — now re-read it, it'll read differently.
- **[Karpathy — nanoGPT](https://github.com/karpathy/nanoGPT)** — the production-quality version of what you just built. 300 lines.
- **[FlashAttention (Dao et al., 2022)](https://arxiv.org/abs/2205.14135)** — makes the `QK^T` matmul IO-efficient. Why modern transformers are 2-4× faster. Full lab on this topic would be its own deep dive.
- **[Rotary Position Embeddings (RoPE, Su et al., 2021)](https://arxiv.org/abs/2104.09864)** — modern replacement for the learned positional embedding we used. Llama, Mistral, GPT-NeoX all use RoPE.
- **[Grouped Query Attention (Ainslie et al., 2023)](https://arxiv.org/abs/2305.13245)** — Llama 2 70B's trick: fewer K and V heads than Q heads. Cuts KV-cache memory at inference.

## What to try next

- Replace learned `pos_emb` with sinusoidal positional encoding (from the original paper) and compare convergence curves.
- Swap standard attention for PyTorch's `F.scaled_dot_product_attention` (uses FlashAttention under the hood) and measure the speedup on a longer sequence.
- Implement **Grouped Query Attention** — reduce the number of K and V heads while keeping Q heads, see if training still converges.